# Apigee LLM Budget Control: Interactive Demo

This notebook guides you through a live demonstration of Apigee's cost-based budget controls for LLM APIs.

We will showcase:
1.  **Metadata-Driven Budgets**: How budgets are defined on API Products without changing proxy code.
2.  **Granular Quota Enforcement**: Real-time cost-based blocking at the gateway.
3.  **Tenant Isolation**: How one team exceeding their budget does not affect other teams.
4.  **Post-Facto Charging**: The "Two-Phase Quota" pattern that handles unknown response costs.

---

## Scenario Design

We have set up two teams:
*   **Team A (Marketing)**: Low budget of **`$0.005`** (5,000 micro-dollars) to quickly trigger the quota block.
*   **Team B (Engineering)**: Higher budget of **`$1.00`** (1,000,000 micro-dollars) for operational continuity.

## Setup & Reset (Run this to start or reset the demo)

Running the cell below will:
1.  Undeploy/Delete any existing demo apps and products.
2.  Recreate them with fresh, empty quota counters.
3.  Load the new API Keys into this notebook's memory.

**You can run this cell at any time during the presentation to reset the demo and start over.**

In [ ]:
import os
import subprocess
import json
import time

# Run the setup script to recreate resources and get new keys
print("Recreating demo resources in Apigee (this may take a few seconds)...")
result = subprocess.run(
    ["bash", "setup-demo-resources.sh"],
    capture_output=True,
    text=True,
    cwd=os.getcwd()
)

if result.returncode != 0:
    print("Error running setup script:")
    print(result.stderr)
    raise RuntimeError("setup-demo-resources.sh failed")

print(result.stdout)

# Load the keys from the generated env file
marketing_key = None
engineering_key = None

with open("demo-keys.env", "r") as f:
    for line in f:
        if "MARKETING_KEY" in line:
            marketing_key = line.split("=")[1].strip().replace('"', '')
        elif "ENGINEERING_KEY" in line:
            engineering_key = line.split("=")[1].strip().replace('"', '')

# Load project ID from env.sh
project_id = None
with open("env.sh", "r") as f:
    for line in f:
        if "export PROJECT=" in line:
            project_id = line.split("=")[1].strip().replace('"', '')

print("--- Loaded Configuration ---")
print(f"Project ID: {project_id}")
print(f"Marketing Key (Budget $0.006): {marketing_key[:8]}...")
print(f"Engineering Key (Budget $1.00): {engineering_key[:8]}...")
print("\nWaiting 30 seconds for API Keys to propagate to Apigee gateways...")
time.sleep(30)
print("Quota Reset Complete! Ready for the demo.")

## Step 1: Engineering Team (Succeeds)

We will send a request using the **Engineering App** (`engineering-app`) calling the expensive `gemini-2.5-pro` model. 
Since this team has a **`$1.00`** budget, the request will succeed.

In [ ]:
!source demo-keys.env && source env.sh && \
  curl -s -D headers.txt -o response.json -X POST "https://$APIGEE_HOST/v2/samples/llm-budget-control/v1/projects/$PROJECT/locations/${REGION:-us-west1}/publishers/google/models/gemini-2.5-pro:generateContent" \
  -H "x-apikey: $ENGINEERING_KEY" \
  -H "x-llm-provider: google" \
  -H "Content-Type: application/json" \
  -d '{ "contents": [{ "role": "user", "parts": [{ "text": "Explain quantum computing in one sentence." }] }] }' && \
  echo "=== HTTP STATUS & CUSTOM HEADERS ===" && \
  grep -E "^(HTTP/|x-llm-)" headers.txt && \
  echo "" && \
  echo "=== RESPONSE CONTENT ===" && \
  jq 'if .fault then "ERROR: " + .fault.faultstring else .candidates[0].content.parts[0].text end' -r response.json

## Step 2: Marketing Team - First Request (Succeeds)

We will send a request using the **Marketing App** (`marketing-app`) calling `gemini-2.5-pro` with a short prompt.
*   The Marketing team's budget is **`$0.02`** (20,000 micro-dollars).
*   Since `gemini-2.5-pro` is a reasoning model, it has a mandatory minimum thinking budget. We set `thinkingBudget: 128` (the minimum allowed) to keep the demo costs low and predictable.
*   The request will succeed and cost around 1,500 micro-dollars.

In [ ]:
!source demo-keys.env && source env.sh && \
  curl -s -D headers.txt -o response.json -X POST "https://$APIGEE_HOST/v2/samples/llm-budget-control/v1/projects/$PROJECT/locations/${REGION:-us-west1}/publishers/google/models/gemini-2.5-pro:generateContent" \
  -H "x-apikey: $MARKETING_KEY" \
  -H "x-llm-provider: google" \
  -H "Content-Type: application/json" \
  -d '{ "contents": [{ "role": "user", "parts": [{ "text": "Write a catchy slogan for a new solar watch." }] }], "generationConfig": { "thinkingConfig": { "thinkingBudget": 128 } } }' && \
  echo "=== HTTP STATUS & CUSTOM HEADERS ===" && \
  grep -E "^(HTTP/|x-llm-)" headers.txt && \
  echo "" && \
  echo "=== RESPONSE CONTENT ===" && \
  jq 'if .fault then "ERROR: " + .fault.faultstring else .candidates[0].content.parts[0].text end' -r response.json

## Step 3: Marketing Team - Second Request (Succeeds, but exhausts budget)

Now we will send a second, larger request using the **Marketing App**.
*   This request asks for a detailed newsletter, which will cost more than the remaining budget.
*   **Note**: Because Apigee uses the **Two-Phase Quota** pattern, it checks the budget in the request path (weight 0) which succeeds. It then calls the LLM, receives the response, and calculates the cost.
*   Even though the cost exceeds the budget, Apigee's response-path policy is configured with `continueOnError="true"`. This is a best practice: since we already incurred the cost on the backend, we allow the user to receive this response.
*   However, the budget counter is now fully exhausted (overdrawn).

In [ ]:
!source demo-keys.env && source env.sh && \
  curl -s -D headers.txt -o response.json -X POST "https://$APIGEE_HOST/v2/samples/llm-budget-control/v1/projects/$PROJECT/locations/${REGION:-us-west1}/publishers/google/models/gemini-2.5-pro:generateContent" \
  -H "x-apikey: $MARKETING_KEY" \
  -H "x-llm-provider: google" \
  -H "Content-Type: application/json" \
  -d '{ "contents": [{ "role": "user", "parts": [{ "text": "Write a 500-word newsletter about the benefits of solar energy." }] }], "generationConfig": { "thinkingConfig": { "thinkingBudget": 128 } } }' && \
  echo "=== HTTP STATUS & CUSTOM HEADERS ===" && \
  grep -E "^(HTTP/|x-llm-)" headers.txt && \
  echo "" && \
  echo "=== RESPONSE CONTENT ===" && \
  jq 'if .fault then "ERROR: " + .fault.faultstring else .candidates[0].content.parts[0].text end' -r response.json

## Step 4: Marketing Team - Third Request (Blocked)

Now we will send a third request using the **Marketing App**, this time trying to call the cheaper **`gemini-2.5-flash`** model.
Even though they are calling a cheaper model, since their budget is already fully exhausted from the previous request, Apigee's request-path check (weight 0) will **immediately block** this request at the gateway, returning a `429 Too Many Requests` error and **preventing any backend costs**.

In [ ]:
!source demo-keys.env && source env.sh && \
  curl -s -D headers.txt -o response.json -X POST "https://$APIGEE_HOST/v2/samples/llm-budget-control/v1/projects/$PROJECT/locations/${REGION:-us-west1}/publishers/google/models/gemini-2.5-flash:generateContent" \
  -H "x-apikey: $MARKETING_KEY" \
  -H "x-llm-provider: google" \
  -H "Content-Type: application/json" \
  -d '{ "contents": [{ "role": "user", "parts": [{ "text": "Explain the greenhouse effect in one paragraph." }] }] }' && \
  echo "=== HTTP STATUS & CUSTOM HEADERS ===" && \
  grep -E "^(HTTP/|x-llm-)" headers.txt && \
  echo "" && \
  echo "=== RESPONSE CONTENT ===" && \
  jq 'if .fault then "ERROR: " + .fault.faultstring else .candidates[0].content.parts[0].text end' -r response.json

## Step 5: Tenant Isolation (Engineering Still Works)

To prove that the budget enforcement is isolated per team (API Product), we will send another request using the **Engineering App**.
Even though the Marketing App is currently blocked, the Engineering App will **succeed** because its budget is independent.

In [ ]:
!source demo-keys.env && source env.sh && \
  curl -s -D headers.txt -o response.json -X POST "https://$APIGEE_HOST/v2/samples/llm-budget-control/v1/projects/$PROJECT/locations/${REGION:-us-west1}/publishers/google/models/gemini-2.5-flash:generateContent" \
  -H "x-apikey: $ENGINEERING_KEY" \
  -H "x-llm-provider: google" \
  -H "Content-Type: application/json" \
  -d '{ "contents": [{ "role": "user", "parts": [{ "text": "Explain the difference between a list and a tuple in Python." }] }] }' && \
  echo "=== HTTP STATUS & CUSTOM HEADERS ===" && \
  grep -E "^(HTTP/|x-llm-)" headers.txt && \
  echo "" && \
  echo "=== RESPONSE CONTENT ===" && \
  jq 'if .fault then "ERROR: " + .fault.faultstring else .candidates[0].content.parts[0].text end' -r response.json

## Wrap-up & Core Concepts to Highlight to the Customer

### 1. The Two-Phase Quota Pattern
LLM costs are dynamic and only known *after* the response is generated. Apigee handles this elegantly:
1.  **Request Phase (Check)**: We call the Quota policy with `weight = 0`. This checks if the user is already over budget. If yes, it blocks them instantly (preventing any backend costs).
2.  **Response Phase (Charge)**: We calculate the actual cost of the generated response (including thinking tokens) and call the Quota policy again with the `weight = actual_cost`. This updates the budget counter.

### 2. Metadata-Driven Budgets
Budgets are defined as custom attributes on the **API Product**. You can change a team's budget in the Apigee UI instantly. There is **no code change** and **no redeployment** required in the API Proxy.

### 3. Business Analytics
Go to the Apigee UI under **Analyze -> Custom Reports** and open the **"LLM Budget and Cost Analysis"** report. You will see:
*   The exact cost and token breakdown by **Developer App** (Team) and **Model**.
*   That the Marketing team's cost stopped accumulating immediately after they were blocked, proving active cost protection.